In [1]:
import sys, os

In [2]:
# notebook parameters, will be replaced by script arguments
application_name = "Blockchain Transaction Count"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
# https://aws.amazon.com/marketplace/pp/prodview-xv4ehzlgtim5a
# aws s3 ls --no-sign-request s3://aws-public-blockchain/v1.0/btc/transactions/ --recursive --summarize
# Total Size: 1816046734815
input_file_1 = "s3a://aws-public-blockchain/v1.0/btc/transactions/date=2009-*" #small sample
output_file_1 = "s3a://uga-data-lake/block_chain_transaction_count_sample"

In [3]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
pyspark.__version__

'4.1.2'

In [4]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)

Running as a notebook
Python version  ['Python 3.12.13']
PySpark version  4.1.2
Java version  ['openjdk 17.0.19 2026-04-21', 'OpenJDK Runtime Environment (build 17.0.19+10-1-deb12u2-Debian)', 'OpenJDK 64-Bit Server VM (build 17.0.19+10-1-deb12u2-Debian, mixed mode, sharing)']


In [5]:
def show(df):
    if interactive:
        df.show()

In [6]:
spark = (SparkSession.builder.appName(application_name)
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-27733761-ae37-4c52-8aeb-4ea219a72fe4;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 110ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

In [7]:
input_data = spark.read.parquet(input_file_1)
input_data.printSchema()
print(input_data.count())

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

root
 |-- txid: string (nullable = true)
 |-- hash: string (nullable = true)
 |-- version: long (nullable = true)
 |-- size: long (nullable = true)
 |-- block_hash: string (nullable = true)
 |-- block_number: long (nullable = true)
 |-- index: long (nullable = true)
 |-- virtual_size: long (nullable = true)
 |-- lock_time: long (nullable = true)
 |-- input_count: long (nullable = true)
 |-- output_count: long (nullable = true)
 |-- is_coinbase: boolean (nullable = true)
 |-- output_value: double (nullable = true)
 |-- outputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- index: long (nullable = true)
 |    |    |-- required_signatures: long (nullable = true)
 |    |    |-- script_asm: string (nullable = true)
 |    |    |-- script_hex: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- value: double (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)
 |-- date: string (nullable = true)
 |-- las

[Stage 2:====================================================>    (11 + 1) / 12]

32709


In [8]:
counts = input_data.groupBy("date").count().orderBy("date")
show(counts)

[Stage 5:====================================================>    (11 + 1) / 12]

+----------+-----+
|      date|count|
+----------+-----+
|2009-01-03|    1|
|2009-01-09|   14|
|2009-01-10|   61|
|2009-01-11|   93|
|2009-01-12|  101|
|2009-01-13|  123|
|2009-01-14|  130|
|2009-01-15|  134|
|2009-01-16|  110|
|2009-01-17|  109|
|2009-01-18|  108|
|2009-01-19|  117|
|2009-01-20|  115|
|2009-01-21|  103|
|2009-01-22|   92|
|2009-01-23|   86|
|2009-01-24|  202|
|2009-01-25|  192|
|2009-01-26|   97|
|2009-01-27|   98|
+----------+-----+
only showing top 20 rows


In [9]:
counts.write.mode("overwrite").csv(output_file_1)

In [10]:
spark.stop()